# Model-Based Collaborative Filtering: Matrix Factorization

**Model-based** = *học* một mô hình nén từ ma trận user-item (bước **train**), sau đó dự đoán chỉ cần mô hình, không cần ma trận gốc nữa.

**Matrix Factorization (MF)** — kỹ thuật thắng giải Netflix Prize — giả định mỗi user và item được mô tả bởi $k$ **yếu tố ẩn** (latent factors). Ví dụ với phim có thể là "độ hành động", "độ lãng mạn"... nhưng máy **tự học ra**, không cần ai đặt tên:

$$R_{(items \times users)} \approx Q_{(items \times k)} \cdot P_{(users \times k)}^T$$

Điểm dự đoán của user $u$ cho item $i$ chỉ là tích vô hướng: $\hat{r}_{i,u} = q_i \cdot p_u$

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Cung ma tran user-item nhu cac notebook truoc
data = {
    'User1': [5, 4, np.nan, 2, 1],
    'User2': [4, 5, 3, np.nan, 2],
    'User3': [np.nan, 2, 5, 4, 3],
    'User4': [3, 4, 2, 5, np.nan],
}
df = pd.DataFrame(data, index=['Item1', 'Item2', 'Item3', 'Item4', 'Item5'])
df

,User1,User2,User3,User4
Item1,5.0,4.0,NaN,3.0
Item2,4.0,5.0,2.0,4.0
Item3,NaN,3.0,5.0,2.0
Item4,2.0,NaN,4.0,5.0
Item5,1.0,2.0,3.0,NaN


## Bước 1: Train bằng Stochastic Gradient Descent

- Khởi tạo $Q$, $P$ ngẫu nhiên.
- Mỗi epoch đi qua **chỉ các ô đã có điểm** (bỏ qua NaN — khác hẳn `fillna(0)`), với mỗi ô:
  - Tính sai số: $e = r_{i,u} - q_i \cdot p_u$
  - Cập nhật ngược hướng gradient: $q_i \mathrel{+}= \eta (e \cdot p_u - \lambda q_i)$, tương tự cho $p_u$
- $\lambda$ (regularization) phạt các hệ số quá lớn để chống overfit — quan trọng vì ta chỉ có 16 điểm dữ liệu.

In [2]:
R = df.to_numpy()                      # 5 items x 4 users
n_items, n_users = R.shape
k = 2                                  # so latent factors
lr, reg, epochs = 0.05, 0.02, 500

# Khoi tao ngau nhien 2 ma tran nho
Q = np.random.normal(0, 0.1, (n_items, k))   # item factors
P = np.random.normal(0, 0.1, (n_users, k))   # user factors

# Danh sach cac o da co diem (training data)
observed = [(i, u) for i in range(n_items) for u in range(n_users)
            if not np.isnan(R[i, u])]
print(f"So o da co diem (training): {len(observed)} / {n_items * n_users}")

for epoch in range(epochs):
    total_err = 0.0
    for i, u in observed:
        err = R[i, u] - Q[i] @ P[u]            # sai so tai o (i, u)
        Q[i] += lr * (err * P[u] - reg * Q[i]) # cap nhat nguoc huong gradient
        P[u] += lr * (err * Q[i] - reg * P[u])
        total_err += err ** 2
    if (epoch + 1) % 100 == 0:
        rmse = np.sqrt(total_err / len(observed))
        print(f"  epoch {epoch+1:>3}: RMSE tren o da biet = {rmse:.4f}")

So o da co diem (training): 16 / 20
  epoch 100: RMSE tren o da biet = 0.3633
  epoch 200: RMSE tren o da biet = 0.3627
  epoch 300: RMSE tren o da biet = 0.3624
  epoch 400: RMSE tren o da biet = 0.3622
  epoch 500: RMSE tren o da biet = 0.3622


## Bước 2: Tái tạo ma trận — dự đoán mọi ô trống

Sau khi train, **mọi ô** (kể cả ô NaN) đều có điểm dự đoán bằng một phép nhân vector — tức thì, không cần ma trận gốc. Đây là lý do model-based mở rộng tốt: **train một lần (offline), phục vụ hàng triệu dự đoán (online)**.

In [3]:
R_hat = pd.DataFrame(Q @ P.T, index=df.index, columns=df.columns)
print("Ma tran tai tao tu mo hinh:")
print(R_hat.round(2))

pred = R_hat.loc['Item3', 'User1']
print(f"\nDu doan User1 cham Item3: {pred:.3f}")

Ma tran tai tao tu mo hinh:
       User1  User2  User3  User4
Item1   4.95   3.82  -1.12   3.15
Item2   3.79   4.92   1.87   4.18
Item3  -0.98   2.63   4.86   2.38
Item4   2.39   5.38   4.29   4.66
Item5   0.48   2.43   2.70   2.13

Du doan User1 cham Item3: -0.982


In [4]:
# Latent factors hoc duoc — may tu rut ra "gu" cua moi user/item
print("User factors P:")
print(pd.DataFrame(P, index=df.columns, columns=['F1', 'F2']).round(2))
print("\nItem factors Q:")
print(pd.DataFrame(Q, index=df.index, columns=['F1', 'F2']).round(2))

User factors P:
         F1    F2
User1  0.84 -2.03
User2 -0.96 -2.29
User3 -2.39 -0.53
User4 -0.89 -1.94

Item factors Q:
         F1    F2
Item1  0.92 -2.05
Item2 -0.34 -2.01
Item3 -1.96 -0.33
Item4 -1.41 -1.76
Item5 -0.99 -0.65


## Vì sao dự đoán có thể ra số âm (ngoài thang 1–5)?

Dự đoán User1–Item3 ra khoảng **-0.98** — không phải bug. MF nhìn toàn cục và nhận ra gu User1 **ngược hẳn** User3 (similarity -0.92 ở `cf_demo.ipynb`), mà User3 lại mê Item3 → mô hình ngoại suy mạnh rằng User1 sẽ ghét Item3, vượt cả thang điểm. Với chỉ 16 điểm dữ liệu, không có gì kìm nó lại.

Hệ thống thực tế khắc phục bằng **bias terms**:

$$\hat{r}_{i,u} = \mu + b_i + b_u + q_i \cdot p_u$$

- $\mu$: điểm trung bình toàn cục
- $b_i$: độ "hot" của item (item hay được chấm cao hơn trung bình)
- $b_u$: độ "dễ tính" của user
- và **clip** kết quả về thang [1, 5]

In [5]:
# Phien ban cai tien: MF + bias terms + clip ve thang [1, 5]
np.random.seed(42)
Qb = np.random.normal(0, 0.1, (n_items, k))
Pb = np.random.normal(0, 0.1, (n_users, k))
b_item = np.zeros(n_items)
b_user = np.zeros(n_users)
mu = np.nanmean(R)                     # diem trung binh toan cuc

for epoch in range(epochs):
    for i, u in observed:
        pred = mu + b_item[i] + b_user[u] + Qb[i] @ Pb[u]
        err = R[i, u] - pred
        b_item[i] += lr * (err - reg * b_item[i])
        b_user[u] += lr * (err - reg * b_user[u])
        Qb[i] += lr * (err * Pb[u] - reg * Qb[i])
        Pb[u] += lr * (err * Qb[i] - reg * Pb[u])

R_hat_bias = mu + b_item[:, None] + b_user[None, :] + Qb @ Pb.T
R_hat_bias = pd.DataFrame(np.clip(R_hat_bias, 1, 5),
                          index=df.index, columns=df.columns)
print("Ma tran tai tao (co bias + clip [1, 5]):")
print(R_hat_bias.round(2))
print(f"\nDu doan User1 cham Item3 (co bias): {R_hat_bias.loc['Item3', 'User1']:.3f}")

Ma tran tai tao (co bias + clip [1, 5]):
       User1  User2  User3  User4
Item1   4.95   4.04   4.55   2.98
Item2   3.99   4.95   2.04   4.00
Item3   4.51   2.98   4.96   2.04
Item4   2.02   3.81   3.99   4.97
Item5   1.04   2.02   3.00   2.83

Du doan User1 cham Item3 (co bias): 4.508


## Nhận xét

**Ưu điểm:**
- **Mở rộng tốt:** train offline một lần, dự đoán online chỉ là tích vô hướng $O(k)$.
- Xử lý data sparsity tốt hơn: latent factors tổng quát hóa từ toàn bộ dữ liệu thay vì so từng cặp.
- Bỏ qua NaN khi train thay vì điền 0 — không bóp méo dữ liệu.

**Nhược điểm:**
- Cần đủ dữ liệu để train, không thì ngoại suy bậy (như ví dụ số âm ở trên).
- Khó giải thích: latent factors là con số trừu tượng, không nói được "vì bạn thích X nên gợi ý Y".
- Phải train lại (hoặc fine-tune) khi có dữ liệu mới — không cập nhật tức thì như memory-based.